# Checkpoint A solutions: a clean agent on a track it was not tuned for

These reference solutions close **Part 1**.
They follow the brief's discipline: look at the training family and plan it on paper (Questions 1–2), tune on `test-circuit` as Unit 7 did (Question 3), find that the result is not clean on a neighbour and look at why (Question 4), tune on the whole family and freeze (Question 5), and only then race the checkpoint track, once, on five seeds (Question 6).
Every race is seeded, so every number below is reproducible, and executable asserts check each answer: the planning answers against an independent derivation, the race answers against the book's pinned values.
Every number is a number about racekit's **carla-like car**: a model, not the truth.

**The brief.**
Your Unit 7 agent is fast on `test-circuit`.
Race day is on a track you have never driven: a variant of `test-circuit`, every corner a little tighter or looser, every straight a little narrower or wider.
You may tune on four other variants of the same family.
You may race the checkpoint track once.

The tracks, all generated by racekit and loaded by name with `load_track`:

- **the training family**, five tracks you may race as often as you like: `test-circuit`, `test-circuit-seed1`, `test-circuit-seed2`, `test-circuit-seed3` and `test-circuit-seed4`;
- **the checkpoint track**, `test-circuit-seed2026`, raced **once**, on race day (Question 6).

The variants differ from `test-circuit` in shape, and the carla-like car may not grip the same on every track, as no two real circuits grip the same.

## The rules

- **One global setting.** You tune `A_LAT` only, on Unit 7's grid (steps of 0.2 m/s²).
- **Everything else is fixed at the Unit 7 reference's values:** `A_DEC = 5.0`, `PP_K = 0.5`, `PROFILE_LEAD_S = 0.3`, and `GRIP` all 1.0.
  Your Unit 7 file may ship the per-section `GRIP` vector you accepted in Unit 7; reset it, because a section is a sixth of a different lap on every track.
  The setup cell's `setting` function writes these fixed values into every variant for you.
- **The clean rule** (Unit 7): a race is clean when it finished, with 0 collisions, and its max |cross-track error| over the driving rows is at most 2 m.
  A setting is clean on a track when it is clean on **race seeds 0–4**.
- **Every race** uses `carla_like(noise=True)` and `max_seconds=450` (a clean lap takes about 320 s).
- **The target:** on the checkpoint track, the median race time over seeds 0–4 is at most **330 s**, with every race clean.
- **No tuning on the checkpoint track.** Loading it and planning on it is allowed; racing it before race day is not.
  Freeze your setting first (Question 5), then race the checkpoint track once, on five seeds (Question 6).

**Your submission** is your own Unit 7 submission.
The setup cell points `SUBMISSION` at `agents/unit-07.py`, the book's Unit 7 reference, as a stand-in: replace that path with your own Unit 7 file before you begin.

Run the setup cell first.
Each question names the variables your answer must define; fill in every `None`, and write each written answer in the markdown cell under its prompt.
Run the questions in order: later questions reuse earlier functions and results.
A race takes a few seconds; no cell should race more than one track, so each cell stays short.

In [ ]:
import math
from pathlib import Path
from tempfile import mkdtemp

import matplotlib.pyplot as plt
import numpy as np
from racekit import write_variant
from racekit.cars import carla_like
from racekit.check_submission import check_submission
from racekit.runner import run_race
from racekit.tracks import grip_sections, load_track

AGENTS = Path("../../projects/racekit/agents")
SUBMISSION = AGENTS / "unit-07.py"  # the book's Unit 7 reference stands in for the student's own file
workdir = Path(mkdtemp(prefix="roar-checkpoint-a-sol-"))

TRAINING = ("test-circuit", "test-circuit-seed1", "test-circuit-seed2", "test-circuit-seed3", "test-circuit-seed4")
CHECKPOINT_TRACK = "test-circuit-seed2026"  # race it once, in Question 6, and never before
SEEDS = (0, 1, 2, 3, 4)  # the clean rule's five race seeds
GRID_STEP = 0.2  # m/s^2: Unit 7's A_LAT grid
FIXED = {"A_DEC": 5.0, "PP_K": 0.5, "PROFILE_LEAD_S": 0.3, "GRIP": (1.0,) * 6}  # the Unit 7 reference's values
A_ACC, TOP_SPEED = 4.0, 45.0  # m/s^2 and m/s: the reference's forward pass and top speed (not tuned here)
MAX_SECONDS = 450.0  # s: every race's time cap
CLEAN_OFFSET = 2.0  # m: the clean rule's limit on the max |cross-track error|
TARGET_S = 330.0  # s: race day's target median
UNIT_6_TIME = 420.40  # s: the Unit 6 agent's median on the checkpoint track, seeds 0-4 (given)
TICK = 0.05 + 1e-9  # s: race times are pinned to one control tick

race_log = []  # every race you run, as (track, seed): your evidence of the sweep


def setting(a_lat):
    # Unit 1's idiom: a copy of SUBMISSION with A_LAT set and the other knobs fixed, written into workdir.
    a_lat = round(a_lat, 1)
    constants = {"A_LAT": a_lat, **FIXED}
    return write_variant(SUBMISSION, workdir / f"a-lat-{a_lat}.py",
                         {key: repr(value) for key, value in constants.items()})


def race(source, track, seed):
    # Every race of this checkpoint: the carla-like car with its noise, one seed, the 450 s cap.
    race_log.append((track, seed))
    return run_race(source, track, car=carla_like(noise=True), seed=seed, max_seconds=MAX_SECONDS)


def max_offset(result):
    # The largest |cross-track error| over the driving rows (held == 0).
    telemetry = result.telemetry
    return float(np.max(np.abs(telemetry["offset"][telemetry["held"] == 0])))


def clean(result):
    # Unit 7's clean rule: finished, 0 collisions, and max |cross-track error| at most 2 m.
    return result.finished and result.collisions == 0 and max_offset(result) <= CLEAN_OFFSET

## Question 1 - Know the family

Before you race anything, look at what you will be tuning on.
Load the five training tracks with `load_track`; each has waypoints `track.x`, `track.y` (one lap, closed: the last waypoint joins waypoint 0), its length `track.length` (m), and the curvature the generator built it with, `track.curvature` (1/m).

For each track, compute from the waypoints yourself:

- the lap length, the sum of the n straight-line segment lengths from waypoint i to waypoint i + 1 (the last segment closes the lap);
- κ_i, Unit 2's Menger curvature of the waypoints (i − 1, i, i + 1), cyclic: 4 × the triangle's area divided by the product of its three sides;
- the tightest radius, 1 / max |κ_i|.

Then compare your curvature with the stored one.

Define:

- `tracks`: a dict from each name in `TRAINING` to its loaded track;
- `menger(xs, ys)`: a function returning a NumPy array of the cyclic Menger curvature at every waypoint (the sign does not matter here, only |κ|);
- `lap_length`: a dict from name to your lap length (m);
- `length_gap`: a dict from name to |your lap length − `track.length`| (m);
- `tightest_radius`: a dict from name to 1 / max |κ_i| from your curvature (m);
- `stored_tightest_radius`: a dict from name to 1 / max |`track.curvature`| (m);
- `curvature_gap`: a dict from name to the largest | |κ_i| − |`track.curvature[i]`| | over the waypoints (1/m);
- `fig`, `ax`: one plot of the five centrelines (one line per track, labelled with its name), with equal axis scales, axis labels, a legend and a title.

In [ ]:
tracks = {name: load_track(name) for name in TRAINING}


def menger(xs, ys):
    # Unit 2's Menger curvature of (i - 1, i, i + 1), cyclic: 4 * area / (product of the three sides).
    n = len(xs)
    before, after = (np.arange(n) - 1) % n, (np.arange(n) + 1) % n
    twice_area = (xs - xs[before]) * (ys[after] - ys[before]) - (ys - ys[before]) * (xs[after] - xs[before])
    sides = (np.hypot(xs - xs[before], ys - ys[before]) * np.hypot(xs[after] - xs, ys[after] - ys)
             * np.hypot(xs[after] - xs[before], ys[after] - ys[before]))
    return 2.0 * twice_area / sides  # 2 * (twice the signed area) = 4 * area


def segment_lengths(xs, ys):
    # segment i runs from waypoint i to waypoint i + 1; the last one closes the lap
    after = (np.arange(len(xs)) + 1) % len(xs)
    return np.hypot(xs[after] - xs, ys[after] - ys)


lap_length, length_gap, tightest_radius, stored_tightest_radius, curvature_gap = {}, {}, {}, {}, {}
for name, track in tracks.items():
    kappa = np.abs(menger(track.x, track.y))
    lap_length[name] = float(np.sum(segment_lengths(track.x, track.y)))
    length_gap[name] = abs(lap_length[name] - track.length)
    tightest_radius[name] = 1.0 / float(np.max(kappa))
    stored_tightest_radius[name] = 1.0 / float(np.max(np.abs(track.curvature)))
    curvature_gap[name] = float(np.max(np.abs(kappa - np.abs(track.curvature))))

fig, ax = plt.subplots(figsize=(8, 8))
for name, track in tracks.items():
    ax.plot(track.x, track.y, label=name, linewidth=1)
ax.set_aspect("equal")
ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
ax.set_title("The training family: five centrelines")
ax.legend()
plt.show()

for name in TRAINING:
    print(f"{name:20s} {len(tracks[name])} waypoints, lap {lap_length[name]:8.2f} m (gap {length_gap[name]:.1e} m), "
          f"tightest radius {tightest_radius[name]:6.3f} m (stored {stored_tightest_radius[name]:6.3f} m), "
          f"largest curvature gap {curvature_gap[name]:.1e} 1/m")
tightest_track = min(tightest_radius, key=tightest_radius.get)
length_range = max(lap_length.values()) - min(lap_length.values())
print(f"tightest corner: {tightest_track}; the lengths span {length_range:.1f} m")

# independent checks: the generator's own lengths and curvature, and the plan's measured tracks (±0.1 m)
assert all(gap < 1e-6 for gap in length_gap.values())
assert all(abs(tightest_radius[name] - stored_tightest_radius[name]) < 0.01 for name in TRAINING)
assert all(gap < 2e-3 for gap in curvature_gap.values())  # a discrete estimate of a smooth curve: close, not equal
pinned = {"test-circuit-seed1": (2625.6, 17.7), "test-circuit-seed2": (2662.0, 18.7),
          "test-circuit-seed3": (2670.4, 20.5), "test-circuit-seed4": (2675.0, 17.3)}
for name, (length, radius) in pinned.items():
    assert abs(lap_length[name] - length) <= 0.1 and abs(stored_tightest_radius[name] - radius) <= 0.1
assert tightest_track == "test-circuit-seed4"
assert 45.0 < length_range < 55.0

**Written answer.**
Which training track has the tightest corner, and how far apart are the five tracks' lengths?
From your plot, in one or two sentences: would you expect a setting that is clean on `test-circuit` to be clean on all of them, and why?

**Answer.**
`test-circuit-seed4` has the tightest corner, a radius of 17.30 m (against 18.00 m on `test-circuit` and 20.46 m, the loosest, on `test-circuit-seed3`).
The lengths run from 2625.6 m (`test-circuit-seed1`) to 2674.9 m (`test-circuit-seed4`), 49.3 m apart, under 2% of a lap.
My Menger curvature matches the stored curvature to within 0.001 1/m at every waypoint, and the tightest radii agree to 0.001 m.
On the plot the five centrelines are one circuit, the same corners in the same order, each a little tighter or looser.
So a setting clean on `test-circuit` will probably be clean on most of them, but not certainly on all: Unit 7's setting sits one grid step inside `test-circuit`'s edge, and a corner 15% tighter, or a car that grips a little less on another track, can use up that one step.

## Question 2 - Plan a lap on paper

Write Unit 7's forward–backward speed profile as a function, so you can plan any track before you race it.
For a closed lap of n waypoints:

1. the cap of every waypoint, min(`top_speed`, √(`a_lat` / |κ_i|)) where |κ_i| > 1e-9, and `top_speed` elsewhere (κ_i your `menger` from Question 1);
2. the backward pass, twice round the lap: for i from n − 1 **down** to 0, v_i = min(v_i, √(v_{(i+1) mod n}² + 2·`a_dec`·Δs_i));
3. then the forward pass, twice round the lap: for i from 0 **up** to n − 1, v_{(i+1) mod n} = min(v_{(i+1) mod n}, √(v_i² + 2·`a_acc`·Δs_i)).

Δs_i is the length of the segment from waypoint i to waypoint i + 1 (the last segment closes the lap).
The planned lap time is the sum over the n segments of Δs_i / ((v_i + v_{(i+1) mod n}) / 2).

Plan each training track at the Unit 7 reference's `A_LAT = 4.6`, with `A_DEC` from `FIXED`, `A_ACC` and `TOP_SPEED`.
Then check every profile against the lesson's two passes:

- **the cap pass:** no waypoint is planned above its cornering cap, v_i ≤ cap_i (to within 1e-9);
- **the backward pass:** no step brakes harder than `A_DEC`, v_i² ≤ v_{(i+1) mod n}² + 2·`A_DEC`·Δs_i (to within 1e-9), at every i including the wrap-around.

Define:

- `lap_profile(xs, ys, a_lat, a_dec, a_acc, top_speed)`: a function returning `(profile, cap, ds)`, three NumPy arrays of n floats;
- `planned_lap_time(profile, ds)`: a function returning the planned time of one lap (s);
- `planned_4_6`: a dict from each name in `TRAINING` to its planned lap time at `A_LAT = 4.6` (s);
- `cap_pass_ok`: a dict from name to `True` when the cap pass check holds at every waypoint;
- `backward_pass_ok`: a dict from name to `True` when the backward pass check holds at every waypoint;
- `fig`, `ax`: one plot of `test-circuit-seed1`'s cap (at most `TOP_SPEED`) and profile against the distance along the lap (`track.s`), with axis labels, a legend and a title.

In [ ]:
def lap_profile(xs, ys, a_lat, a_dec, a_acc, top_speed):
    n = len(xs)
    kappa = menger(xs, ys)
    ds = segment_lengths(xs, ys)
    cap = np.array([min(top_speed, math.sqrt(a_lat / abs(k))) if abs(k) > 1e-9 else top_speed for k in kappa])
    v = cap.tolist()
    for _ in range(2):  # the backward pass, twice round the lap
        for i in range(n - 1, -1, -1):
            v[i] = min(v[i], math.sqrt(v[(i + 1) % n] ** 2 + 2.0 * a_dec * ds[i]))
    for _ in range(2):  # the forward pass, twice round the lap
        for i in range(n):
            v[(i + 1) % n] = min(v[(i + 1) % n], math.sqrt(v[i] ** 2 + 2.0 * a_acc * ds[i]))
    return np.array(v), cap, ds


def planned_lap_time(profile, ds):
    after = np.roll(profile, -1)  # v_{(i+1) mod n}
    return float(np.sum(ds / ((profile + after) / 2.0)))


planned_4_6, cap_pass_ok, backward_pass_ok, forward_pass_ok = {}, {}, {}, {}
for name, track in tracks.items():
    profile, cap, ds = lap_profile(track.x, track.y, 4.6, FIXED["A_DEC"], A_ACC, TOP_SPEED)
    after = np.roll(profile, -1)
    planned_4_6[name] = planned_lap_time(profile, ds)
    cap_pass_ok[name] = bool(np.all(profile <= cap + 1e-9))
    backward_pass_ok[name] = bool(np.all(profile ** 2 <= after ** 2 + 2.0 * FIXED["A_DEC"] * ds + 1e-9))
    forward_pass_ok[name] = bool(np.all(after ** 2 <= profile ** 2 + 2.0 * A_ACC * ds + 1e-9))

seed1 = tracks["test-circuit-seed1"]
seed1_profile, seed1_cap, _ = lap_profile(seed1.x, seed1.y, 4.6, FIXED["A_DEC"], A_ACC, TOP_SPEED)
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(seed1.s, seed1_cap, color="lightgrey", label="the cap, sqrt(4.6 / |kappa|), at most 45 m/s")
ax.plot(seed1.s, seed1_profile, label="the profile (both passes, twice round)")
ax.set_xlabel("distance along the lap, s (m)")
ax.set_ylabel("speed (m/s)")
ax.set_title("test-circuit-seed1 at A_LAT 4.6: the cap and the planned profile")
ax.legend()
plt.show()

for name in TRAINING:
    print(f"{name:20s} planned lap {planned_4_6[name]:7.3f} s (three laps {3 * planned_4_6[name]:7.2f} s); "
          f"cap pass {cap_pass_ok[name]}, backward pass {backward_pass_ok[name]}, forward pass {forward_pass_ok[name]}")

# independent checks: the two passes hold, the slowest point is a cap, and a third lap of the passes changes nothing
assert all(cap_pass_ok.values()) and all(backward_pass_ok.values()) and all(forward_pass_ok.values())
assert abs(float(np.min(seed1_profile)) - float(np.min(seed1_cap))) < 1e-12
assert abs(float(np.min(seed1_cap)) - math.sqrt(4.6 * tightest_radius["test-circuit-seed1"])) < 1e-9
assert all(100.0 < planned_4_6[name] < 106.0 for name in TRAINING)
assert max(planned_4_6.values()) - min(planned_4_6.values()) < 1.5  # the family plans within about a second

**Written answer.**
A race is three laps, so a planned lap is not a race time.
In one or two sentences: name one thing the plan assumes that the race will not give it.

**Answer.**
The plan assumes a flying lap: a closed loop whose first waypoint is entered at the speed the last one leaves, so it never pays for the race's standing start from 0 m/s on lap 1.
It also assumes the car drives exactly the planned speed on the centreline; in the race the speed loop chases a goal read 0.3 s ahead and pure pursuit drives its own line, so the car is sometimes above and sometimes below the plan (Question 6 measures which wins).

## Question 3 - Tune on one track

Do what Unit 7 did, on `test-circuit` alone.

First write the clean rule on five seeds as a function.
`five_seed_check(source, track)` races `source` on `track` with `race`, on the seeds in `SEEDS` in order, and **stops at the first race that is not clean** (one failing race is enough to call the setting not clean, and the races after it would only cost time).
It returns `(all_clean, results)`: `all_clean` is `True` when all five races were clean, and `results` is the list of the races it ran.

Then find `test-circuit`'s edge on the grid: start at your Unit 7 submission's `A_LAT` (the reference ships 4.6) and step **up** by `GRID_STEP`, one `setting(a_lat)` variant per value, until a value is not clean.
The edge is the last clean value; the margin rule (Unit 7) ships one grid step inside it.
(If your starting value is itself not clean, step down instead until you find a clean one.)

Race only `test-circuit` in this cell.

Define:

- `five_seed_check(source, track)`: the function;
- `one_track_sweep`: a dict from each `A_LAT` you raced (rounded to 1 decimal) to `(all_clean, median)`, with `median` the median race time over seeds 0–4 (s) when `all_clean`, and `None` otherwise;
- `one_track_edge`: the highest clean value you found;
- `one_track_margin`: the margin rule's value, `one_track_edge - GRID_STEP`, rounded to 1 decimal.

In [ ]:
def five_seed_check(source, track):
    results = []
    for seed in SEEDS:
        result = race(source, track, seed)
        results.append(result)
        if not clean(result):
            return False, results  # one failing race is enough
    return True, results


def median_time(all_clean, results):
    return float(np.median([result.total_time for result in results])) if all_clean else None


def shown(median):
    return "-" if median is None else f"{median:.2f} s"


START_A_LAT = 4.6  # the A_LAT the Unit 7 reference ships
one_track_sweep, one_track_results = {}, {}
a_lat, direction = START_A_LAT, None
while True:
    all_clean, results = five_seed_check(setting(a_lat), "test-circuit")
    one_track_sweep[a_lat], one_track_results[a_lat] = (all_clean, median_time(all_clean, results)), results
    if direction is None:
        direction = 1 if all_clean else -1  # up from a clean start, down from an unclean one
    if (direction == 1 and not all_clean) or (direction == -1 and all_clean):
        break
    a_lat = round(a_lat + direction * GRID_STEP, 1)

one_track_edge = max(value for value, (all_clean, _) in one_track_sweep.items() if all_clean)
one_track_margin = round(one_track_edge - GRID_STEP, 1)

for value, (all_clean, median) in one_track_sweep.items():
    last = one_track_results[value][-1]
    print(f"A_LAT {value}: clean {all_clean}, {len(one_track_results[value])} races, median {shown(median)}; "
          f"last race seed {last.seed}: finished {last.finished}, {last.collisions} collisions, "
          f"max |offset| {max_offset(last):.2f} m")
print(f"test-circuit's edge {one_track_edge}; the margin rule ships {one_track_margin}")

# the book's pins (Unit 7's grid, re-raced here at 450 s)
assert list(one_track_sweep) == [4.6, 4.8, 5.0]
assert one_track_sweep[4.6][0] and abs(one_track_sweep[4.6][1] - 304.15) <= TICK
assert one_track_sweep[4.8][0] and abs(one_track_sweep[4.8][1] - 300.90) <= TICK
assert one_track_sweep[5.0] == (False, None) and len(one_track_results[5.0]) == 1
assert one_track_edge == 4.8 and one_track_margin == 4.6
assert len(race_log) == 11 and all(track == "test-circuit" for track, _ in race_log)

## Question 4 - Check the family

A setting tuned on one track has only been tested on that track.
Race `one_track_margin` on the four variants, `test-circuit-seed1` to `test-circuit-seed4`, with `five_seed_check`: one track per cell, below.

Define:

- `family_check`: a dict from each variant's name to the `(all_clean, results)` that `five_seed_check(setting(one_track_margin), name)` returned;
- `failing_track`: the name of the first variant (in the order of `TRAINING`) on which `one_track_margin` is not clean (`None` if it is clean on all four);
- `failing_race`: the race that was not clean on `failing_track` (the last race in its `results`).

In [ ]:
family_check = {}


def check_variant(name):
    family_check[name] = five_seed_check(setting(one_track_margin), name)
    all_clean, results = family_check[name]
    print(f"A_LAT {one_track_margin} on {name}: clean {all_clean}, {len(results)} races, "
          f"median {shown(median_time(all_clean, results))}")
    for result in results:
        print(f"  seed {result.seed}: {result.total_time:.2f} s, finished {result.finished}, "
              f"{result.collisions} collisions, max |offset| {max_offset(result):.2f} m, clean {clean(result)}")
    return all_clean, results


all_clean, results = check_variant("test-circuit-seed1")

# the book's pin: 4.6 is not clean on seed 1, and race seed 1 is the race that fails
assert not all_clean and len(results) == 2
assert clean(results[0]) and not clean(results[1]) and results[1].seed == 1

In [ ]:
all_clean, results = check_variant("test-circuit-seed2")
assert all_clean and len(results) == 5

In [ ]:
all_clean, results = check_variant("test-circuit-seed3")
assert all_clean and len(results) == 5

In [ ]:
all_clean, results = check_variant("test-circuit-seed4")
assert all_clean and len(results) == 5

failing_track = next((name for name in TRAINING[1:] if not family_check[name][0]), None)
failing_race = family_check[failing_track][1][-1] if failing_track else None
print(f"failing track {failing_track}, race seed {failing_race.seed}: {failing_race}")
assert failing_track == "test-circuit-seed1" and failing_race.seed == 1
assert sum(len(family_check[name][1]) for name in TRAINING[1:]) == 2 + 5 + 5 + 5

Now look at the failing race.
Its first collision happens at `failing_race.respawn_times[0]` (Unit 1).
Plot what the car did in the 10 s before it.

Define:

- `first_collision_t`: the time of the failing race's first collision (s);
- `window`: a boolean array over the failing race's telemetry rows, `True` for the rows with `first_collision_t - 10 <= t <= first_collision_t`;
- `fig`, `axes`: one figure with two plots, one above the other with a shared time axis: the cross-track error (`offset`, m) against `t`, and the speed (`v`, m/s) against `t`, over `window`; with axis labels and a title naming the track, the `A_LAT` and the race seed;
- `offset_every_half_second`: a list of the cross-track error (m) at the rows every 0.5 s through `window`, from its first row to its last (a telemetry row is one control tick of 0.05 s, so every 10th row).

In [ ]:
telemetry = failing_race.telemetry
first_collision_t = failing_race.respawn_times[0]
window = (telemetry["t"] >= first_collision_t - 10.0) & (telemetry["t"] <= first_collision_t)
rows = np.flatnonzero(window)
offset_every_half_second = [float(value) for value in telemetry["offset"][rows[::10]]]

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].plot(telemetry["t"][window], telemetry["offset"][window])
axes[0].axhline(CLEAN_OFFSET, color="grey", linestyle="--", linewidth=1)
axes[0].axhline(-CLEAN_OFFSET, color="grey", linestyle="--", linewidth=1)
axes[0].set_ylabel("cross-track error (m)")
axes[1].plot(telemetry["t"][window], telemetry["v"][window])
axes[1].set_ylabel("speed (m/s)")
axes[1].set_xlabel("t (s)")
axes[0].set_title(f"{failing_track}, A_LAT {one_track_margin}, race seed {failing_race.seed}: "
                  f"the 10 s before the first collision")
plt.show()

# where the car was: the waypoint under each sampled row (Track.locate), and the track's radius there
track = tracks[failing_track]
kappa = np.abs(menger(track.x, track.y))
sections = grip_sections(track)
n = len(track)
samples = rows[::10]
sample_waypoints = [track.locate(telemetry["x"][row], telemetry["y"][row])[0] for row in samples]
print(f"first collision at t = {first_collision_t:.2f} s, on lap {telemetry['progress'][rows[-1]] // n + 1} of 3")
print("    t (s)  offset (m)  v (m/s)  waypoint  section  radius there (m)")
for row, waypoint in zip(samples, sample_waypoints):
    radius = 1.0 / kappa[waypoint] if kappa[waypoint] > 1e-9 else math.inf
    print(f"  {telemetry['t'][row]:7.2f}  {telemetry['offset'][row]:+9.2f}  {telemetry['v'][row]:7.2f}  "
          f"{waypoint:8d}  {sections[waypoint]:7d}  {radius:12.1f}")

# the corner before the weave: walk back from the waypoint 4 s before the collision to the last one tighter than 100 m
weave_start = sample_waypoints[-9]
k = weave_start
while kappa[k] < 1.0 / 100.0:
    k = (k - 1) % n
corner_exit = k
corner = []
while kappa[k] >= 1.0 / 100.0:  # and on back through the corner to its entry
    corner.append(k)
    k = (k - 1) % n
corner_radius = 1.0 / float(np.max(kappa[corner]))
straight_run = (track.s[weave_start] - track.s[corner_exit]) % track.length
weave_speeds = telemetry["v"][samples[-9:]]
print(f"the corner before ends at waypoint {corner_exit} (radius down to {corner_radius:.1f} m), "
      f"{straight_run:.0f} m before the last 4 s begin")
print(f"speed over the last 4 s {float(np.min(weave_speeds)):.2f}..{float(np.max(weave_speeds)):.2f} m/s; "
      f"pure pursuit's lookahead PP_K * v = {FIXED['PP_K'] * float(np.max(weave_speeds)):.1f} m")
print("cross-track error every 0.5 s (m):", " ".join(f"{value:+.2f}" for value in offset_every_half_second))

# the weave: calm, then a growing oscillation that crosses 1 m and then 2 m in the last 4 s
last_4_s = offset_every_half_second[-9:]
assert len(offset_every_half_second) == 21
assert max(abs(value) for value in offset_every_half_second[:-9]) < 1.0
assert max(abs(value) for value in last_4_s) > CLEAN_OFFSET
assert min(last_4_s) < -1.0 and max(last_4_s) > 1.0  # it swings to both sides, not a drift to one wall
assert all(kappa[waypoint] < 1.0 / 100.0 for waypoint in sample_waypoints[-9:])  # on a straight
assert float(np.min(weave_speeds)) > 35.0 and float(np.max(weave_speeds)) > TOP_SPEED  # at top speed, accelerating out

**Written answer.**
Answer in three parts.

1. **Describe** what the car did in those 10 s: where it was (on a corner or a straight, and after what), how fast it was going, and what the cross-track error did, with numbers from your plot.
2. **Propose a cause** in the terms of Units 5–6: for example the pure-pursuit lookahead at that speed, the grip the car had there, or the corner before.
   You are graded on the description and the reasoning, not on finding one particular mechanism.
3. **Name one race** (one setting, on one training track, on one seed) that would test your proposed cause, and say what result would support it and what result would count against it.
   You do not have to run it.

**Answer.**

1. **Describe.** The failing race is `test-circuit-seed1`, A_LAT 4.6, race seed 1, and the first collision comes at t = 207.10 s, on lap 3, about 86 waypoints (some 170 m) after the start line.
   Ten seconds earlier the car is in the lap's last corner, radius 65.4 m, at 18.0 m/s; it leaves the corner at waypoint 1242 and accelerates down the start straight (curvature 0 under the car for the last 8 s), reaching 45.3 m/s at t = 204.6 s and holding about 45.5–46.0 m/s to the wall.
   The cross-track error is a weave, not a drift: it swings from side to side with a period of about 2.5 s, and each swing is larger than the last: about 0.7 m out of the corner, then 0.8, 0.9, 1.3 and 2.5 m (+2.47 m at t = 205.6 s, at 45.9 m/s), and the next swing goes the other way and hits the wall (−1.36 m at t = 206.6 s, −6.02 m at the collision).
   It is calm (under 1 m) for the first 6 s and crosses 1 m and then 2 m in the last 4 s, while the speed climbs from 39 to 46 m/s.
2. **Propose a cause.** The weave grows as the speed grows, so my first candidate is pure pursuit at speed (Unit 5): at 46 m/s the lookahead is PP_K · v = 23 m, and the car covers more than 2 m in each control tick, so a steering correction made a tick or two late (the car's delay, Unit 4) overshoots, and the next correction overshoots the other way; a lookahead that was long enough to damp this at 40 m/s may not be at 46 m/s.
   The other candidates are the grip (Unit 6): if the car grips a little less on this track, the same steering correction swings it further, and the margin `test-circuit` gave us is gone; and the corner before: the car leaves it already 0.7 m off the line and swinging, so the weave was seeded in the corner, and A_LAT (the only thing that changed from a clean setting) changes how fast and how cleanly the car leaves it.
   The evidence here does not prove one of these; 4.4 being clean on the same track (Question 5) says only that a slower corner keeps the weave inside 2 m.
3. **A race that would test it.** Race `test-circuit-seed1`, race seed 1, at A_LAT 4.6 with PP_K 0.6 instead of 0.5 (a longer lookahead at the same speed).
   If the lookahead is the cause, the weave on the start straight should shrink and the race should be clean; if the car still weaves to the wall at the same place with the same growth, that counts against the lookahead and points at the grip or the corner exit.

## Question 5 - Tune on the family

Now tune on the whole family: a setting counts as clean only when it is clean on **all five** training tracks.
Find the highest grid value that is, apply the margin rule, and freeze it.

Use what you have already raced: Questions 3 and 4 tell you which values are already not clean somewhere.
Step down the grid from there, racing one track per cell, below; stop racing a value as soon as one track is not clean.

Define:

- `family_sweep`: a dict from `(a_lat, name)` (with `a_lat` rounded to 1 decimal) to `(all_clean, median)` for every value and track you raced in this question, `median` as in Question 3;
- `family_edge`: the highest grid value clean on all five training tracks;
- `FROZEN_A_LAT`: the margin rule's value, `family_edge - GRID_STEP`, rounded to 1 decimal;
- `FROZEN_SUBMISSION`: `setting(FROZEN_A_LAT)`, the file you will race on race day;
- `frozen_report`: `check_submission(FROZEN_SUBMISSION)`, with `frozen_report.ok` `True`.

After this question your setting is frozen: nothing about it may change once you have raced the checkpoint track.

In [ ]:
# Values already shown not clean somewhere: 5.0 on test-circuit (Question 3) and 4.6 on seed 1 (Question 4).
not_clean_somewhere = [value for value, (all_clean, _) in one_track_sweep.items() if not all_clean]
not_clean_somewhere += [one_track_margin for name in TRAINING[1:] if not family_check[name][0]]
candidate = round(min(not_clean_somewhere) - GRID_STEP, 1)  # the highest value not yet shown unclean
family_sweep = {}


def check_family_track(name):
    # race the candidate on one track, unless it has already failed on an earlier track
    if not all(all_clean for (value, _), (all_clean, _) in family_sweep.items() if value == candidate):
        print(f"A_LAT {candidate} already not clean on the family: {name} not raced")
        return
    all_clean, results = five_seed_check(setting(candidate), name)
    family_sweep[(candidate, name)] = (all_clean, median_time(all_clean, results))
    print(f"A_LAT {candidate} on {name}: clean {all_clean}, {len(results)} races, "
          f"median {shown(family_sweep[(candidate, name)][1])}, max |offset| {max(max_offset(r) for r in results):.2f} m")


check_family_track("test-circuit")
assert candidate == 4.4

In [ ]:
check_family_track("test-circuit-seed1")

In [ ]:
check_family_track("test-circuit-seed2")

In [ ]:
check_family_track("test-circuit-seed3")

In [ ]:
check_family_track("test-circuit-seed4")

In [ ]:
clean_on_all = [value for value in {value for value, _ in family_sweep}
                if all(family_sweep.get((value, name), (False, None))[0] for name in TRAINING)]
family_edge = max(clean_on_all)
FROZEN_A_LAT = round(family_edge - GRID_STEP, 1)
FROZEN_SUBMISSION = setting(FROZEN_A_LAT)
frozen_report = check_submission(FROZEN_SUBMISSION)
print(frozen_report)
print(f"family edge {family_edge}; frozen A_LAT {FROZEN_A_LAT}")

# the book's pins: 4.4 clean on all five at these medians, so the margin rule freezes 4.2
pinned_4_4 = {"test-circuit": 307.55, "test-circuit-seed1": 305.40, "test-circuit-seed2": 307.15,
              "test-circuit-seed3": 308.15, "test-circuit-seed4": 308.05}
for name, median in pinned_4_4.items():
    assert family_sweep[(4.4, name)][0] and abs(family_sweep[(4.4, name)][1] - median) <= TICK
assert family_edge == 4.4 and FROZEN_A_LAT == 4.2 and frozen_report.ok
assert set(family_sweep) == {(4.4, name) for name in TRAINING}  # 4.2 is not raced on the family here
assert all(track != CHECKPOINT_TRACK for track, _ in race_log)  # the checkpoint track is still unraced
assert len(race_log) == 11 + 17 + 25

**Written answer.**
Unit 7 already used a margin of one grid step on `test-circuit`.
In two or three sentences: why was that margin not enough for the family, and what does the margin you applied here protect you against that the family sweep cannot?

**Answer.**
Unit 7's margin was measured on one track: it protects against the noise between race seeds on `test-circuit`, and on `test-circuit` it held (4.6 clean, the edge 4.8).
But a neighbouring track's edge can sit lower than one track's margin: `test-circuit-seed1`'s corners and its grip are different, and there 4.6 is already past the edge, so the difference between tracks is bigger than one grid step.
The margin applied here (4.2, one step inside the family's edge 4.4) protects against what the family sweep cannot measure: a track that is not in the family, whose edge may sit below the family's (the checkpoint track, never raced before race day), and race seeds beyond 0–4; it costs 4.20 s per race on paper (Question 6).

## Question 6 - Race day

Your setting is frozen.
**First plan, then race.**

Load the checkpoint track (allowed) and plan it with Question 2's `lap_profile` and `planned_lap_time`, at `FROZEN_A_LAT` and the fixed knobs.
Also plan it one grid step higher, at `FROZEN_A_LAT + GRID_STEP` (planning is allowed; racing that value here is not): the difference is what the margin costs on paper.

Define:

- `checkpoint_track`: `load_track(CHECKPOINT_TRACK)`;
- `planned_lap_frozen`: the planned lap time at `FROZEN_A_LAT` (s);
- `planned_race_frozen`: the planned time of a three-lap race, `3 * planned_lap_frozen` (s);
- `planned_margin_cost`: the planned lap time at `FROZEN_A_LAT` minus the planned lap time at `FROZEN_A_LAT + GRID_STEP`, times 3 (s per race).

In [ ]:
checkpoint_track = load_track(CHECKPOINT_TRACK)
xs, ys = checkpoint_track.x, checkpoint_track.y
frozen_profile, _, ds = lap_profile(xs, ys, FROZEN_A_LAT, FIXED["A_DEC"], A_ACC, TOP_SPEED)
higher_profile, _, _ = lap_profile(xs, ys, round(FROZEN_A_LAT + GRID_STEP, 1), FIXED["A_DEC"], A_ACC, TOP_SPEED)
planned_lap_frozen = planned_lap_time(frozen_profile, ds)
planned_lap_higher = planned_lap_time(higher_profile, ds)
planned_race_frozen = 3 * planned_lap_frozen
planned_margin_cost = 3 * (planned_lap_frozen - planned_lap_higher)
checkpoint_tightest = 1.0 / float(np.max(np.abs(menger(xs, ys))))

print(f"{checkpoint_track}: tightest radius {checkpoint_tightest:.1f} m")
print(f"planned lap at A_LAT {FROZEN_A_LAT}: {planned_lap_frozen:.3f} s; three laps {planned_race_frozen:.2f} s")
print(f"planned lap at A_LAT {round(FROZEN_A_LAT + GRID_STEP, 1)}: {planned_lap_higher:.3f} s; "
      f"the margin costs {planned_margin_cost:.2f} s per race on paper")

# independent checks: the plan's measured track, and a lower A_LAT never plans faster
assert abs(checkpoint_track.length - 2655.8) <= 0.1 and abs(checkpoint_tightest - 19.8) <= 0.1
assert planned_margin_cost > 0.0 and 3.0 < planned_margin_cost < 6.0
assert planned_race_frozen < TARGET_S
assert all(track != CHECKPOINT_TRACK for track, _ in race_log)  # planned, not raced

Now race `FROZEN_SUBMISSION` on the checkpoint track, **once**, on every seed in `SEEDS`.
Race all five seeds even if one is not clean: race day is the whole result.
Run this cell once.

Define:

- `race_day`: the list of the five races, in seed order (race each with `race`, not with `five_seed_check`, which stops early);
- `race_day_times`: the list of their race times (s);
- `race_day_clean`: `True` when all five races are clean;
- `race_day_median`, `race_day_spread`: the median and the spread (max − min) of the times (s);
- `within_target`: `True` when `race_day_clean` and `race_day_median <= TARGET_S`;
- `race_minus_plan`: `race_day_median - planned_race_frozen` (s);
- `gain_over_unit_6`: `UNIT_6_TIME - race_day_median` (s);
- `checkpoint_races`: the number of entries in `race_log` on `CHECKPOINT_TRACK` (it should be exactly 5).

In [ ]:
race_day = [race(FROZEN_SUBMISSION, CHECKPOINT_TRACK, seed) for seed in SEEDS]
race_day_times = [result.total_time for result in race_day]
race_day_clean = all(clean(result) for result in race_day)
race_day_median = float(np.median(race_day_times))
race_day_spread = max(race_day_times) - min(race_day_times)
within_target = race_day_clean and race_day_median <= TARGET_S
race_minus_plan = race_day_median - planned_race_frozen
gain_over_unit_6 = UNIT_6_TIME - race_day_median
checkpoint_races = sum(1 for track, _ in race_log if track == CHECKPOINT_TRACK)

for result in race_day:
    print(f"seed {result.seed}: {result.total_time:.2f} s, finished {result.finished}, "
          f"{result.collisions} collisions, max |offset| {max_offset(result):.2f} m, clean {clean(result)}")
print(f"clean {race_day_clean}; median {race_day_median:.2f} s, spread {race_day_spread:.2f} s; "
      f"within the {TARGET_S:.0f} s target: {within_target}")
print(f"race minus plan {race_minus_plan:+.2f} s; gain over Unit 6 {gain_over_unit_6:.2f} s; "
      f"checkpoint races in the log: {checkpoint_races}")

# why the race and the plan differ: lap by lap, and the car's speed against the plan (seed 0)
n = len(checkpoint_track)
telemetry = race_day[0].telemetry
lap_of_row = telemetry["progress"] // n
lap_ends = [0.0] + [float(np.max(telemetry["t"][lap_of_row <= lap])) for lap in range(3)]
lap_times = [lap_ends[lap + 1] - lap_ends[lap] for lap in range(3)]
flying = (lap_of_row >= 1) & (telemetry["held"] == 0)
above_plan = float(np.mean(telemetry["v"][flying] - frozen_profile[telemetry["progress"][flying] % n]))
print("seed 0 lap times (s):", " ".join(f"{t:.2f}" for t in lap_times), f"against a planned lap of {planned_lap_frozen:.2f} s")
print(f"on laps 2-3 the car runs {above_plan:+.2f} m/s from the plan on average; "
      f"top speed {float(np.max(telemetry['v'])):.2f} m/s against the plan's {TOP_SPEED:.0f}")

# the book's pins: clean on every seed, median 310.80 s, within the 330 s target, raced exactly once
assert race_day_clean and abs(race_day_median - 310.80) <= TICK and within_target
assert race_day_median <= TARGET_S < UNIT_6_TIME
assert checkpoint_races == 5 and len(race_log) == 11 + 17 + 25 + 5
assert abs(gain_over_unit_6 - (UNIT_6_TIME - 310.80)) <= TICK

**Written answer.**
In three to five sentences, report race day: the median and spread, the clean rule and the target.
Compare the race with your plan: is the race faster or slower than the plan, by how much, and why? Is the gap what you would expect from Question 2's answer?
Compare it with the Unit 6 agent's 420.40 s: what did speed planning buy?
And what did the margin cost, on paper (`planned_margin_cost`), against what it bought you on a track you had never raced?

**Answer.**
Race day: all five races are clean (0 collisions, max |cross-track error| at most 1.44 m), with a median of 310.80 s and a spread of 0.20 s, so the agent meets the 330 s target with 19.2 s to spare.
The race is **faster** than the plan: three planned laps are 316.96 s, and the race is 6.16 s quicker.
The standing start does cost what Question 2 said (lap 1 takes 107.00 s against a planned 105.65 s), but laps 2 and 3 take 102.00 s and 101.80 s, about 3.7 s faster than the plan each, because the car runs on average 0.79 m/s above its own profile: it reads the goal 0.3 s ahead, so it accelerates out of every corner before the plan does, and its speed loop lets it overshoot the 45 m/s top speed to 46.1 m/s.
So the plan is a good estimate (within 2%) but not a lower bound on the race time, and the sign of the gap is something only a race can show.
Against the Unit 6 agent's 420.40 s, speed planning bought 109.60 s, a quarter of the race; the margin cost 4.20 s per race on paper (Question 6's plan at 4.2 against 4.4), about 1% of the race, and bought a setting one grid step inside the family's edge on a track we had never raced, the same kind of step that `test-circuit`'s margin turned out to need on `test-circuit-seed1`.